In [1]:
# TODO make sure this renders correctly on repo

# Optimizer & Learning Rate Schedule

**Llama 3 Pre-training:** 
- "We pre-train **Llama 3 405B** using **AdamW** with a **peak learning rate of** $\mathbf{8 × 10^{−5}}$ , **a linear warm up of 8,000 steps**, and **a cosine learning rate schedule decaying to** $\mathbf{8 × 10^{−7}}$ **over** $\mathbf{1{,}200{,}000}$ **steps**." 
- During pre-training on the final $40\text{M}$ tokens, we **linearly annealed the learning rate** to $0$

**Llama 2 paper:**
- Hyperparameters. We trained using the **AdamW** optimizer (Loshchilov and Hutter, 2017), with $\boldsymbol{\beta_1 = 0.9}$, $\boldsymbol{\beta_2 = 0.95}$, $\cancel{eps = 10^{−5}}$. ~~We use a cosine learning rate schedule~~, ~~with warmup of~~ $\cancel{2000}$ ~~steps, and decay final learning rate down to 10% of the peak learning rate**. We use a weight decay of $\cancel{0.1}$ and gradient clipping of 1.0. Figure 5 (a) shows the training loss for Llama 2 with these hyperparameters.~~

*The items like $\beta_1$ aren't mentioned in the third paper, so they are referenced in the second paper*

In [2]:
import easyjupyter
import torch
import torch.nn as nn

In [3]:
import math
import torch.optim as optim
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from configs import ConfigTemplate

In [5]:
def get_optimizer(cfg: ConfigTemplate, model: nn.Module):
    """Llama 3 did not specify the betas, we assume the following betas from the llama 2 paper."""
    return optim.AdamW(
        model.parameters(),
        lr=cfg.text_only.pretrain.initial_stage.peak_lr,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=0.1,
    )

## Test: Show Learning Rate

In [7]:
# @i-c
from configs import Llama3_1_405B
import math

cfg = Llama3_1_405B.initialize()
initial_cfg = cfg.text_only.pretrain.initial_stage
lc_cfg = cfg.text_only.pretrain.lc_stage
annealing_cfg = cfg.text_only.pretrain.annealing_stage


print("\n------ LR Schedule Pre-Training Example For The Llama 3 405B Model ------\n")
print(f"Peak LR: {initial_cfg.peak_lr
                  :.8f} | Min LR: {initial_cfg.decay_lr_to:.8f}")
print(f"Initial Stage Steps: {initial_cfg.steps:,}")
print(f"    Warmup ends at step: {initial_cfg.warmup_steps:,}")
print(f"Long-Context Stage Tokens: {lc_cfg.tokens:,}")
print(f"Annealing Stage Tokens: {annealing_cfg.tokens:,}")
print(f"\n" + "-" * 84)
print(
    f"{'Stage':<15} | {'Metric (Step/Token)':<22} | {'Learning Rate':<15} | {'Phase'}"
)
print("-" * 84)

initial_milestones = [
    1,  # Start of warmup
    initial_cfg.warmup_steps // 2,  # halfway through warmup
    initial_cfg.warmup_steps,  # Warm up ends peak learning rate
    initial_cfg.steps // 2,  # Mid cosine decay
    initial_cfg.steps - 1,  # Near end of initial stage
    initial_cfg.steps,  # End of the initial stage 1,200,000 steps
]
lr = None

for step in initial_milestones:
    initial_cfg.steps_completed = step
    lr = initial_cfg.get_learning_rate(initial_min_lr=initial_cfg.decay_lr_to)

    if initial_cfg.steps_completed <= initial_cfg.warmup_steps:
        phase = "Warmup (Linear LR up ↑)"
    else:
        phase = "Decay (Cosine LR down ↓)"

    print(f"{'Initial':<15} | Step {initial_cfg.steps_completed:<17,d} | {lr:15.8f} | {phase}")

print("\n\n" + "-" * 84)

lc_milestones = [
    1,  # Start of long-context stage
    lc_cfg.tokens // 2,  # Mid long-context stage
    lc_cfg.tokens,  # End of long-context stage
]

for tokens in lc_milestones:
    lc_cfg.tokens_processed = tokens
    lr = lc_cfg.get_learning_rate(initial_min_lr=initial_cfg.decay_lr_to)
    phase = "Flat at Min LR (→)"
    print(f"{'Long-Context':<15} | Token {lc_cfg.tokens_processed:<16,d} | {lr:15.8f} | {phase}")

print("\n\n" + "-" * 84)


annealing_milestones = [
    1,  # Start of annealing
    annealing_cfg.tokens // 2,  # mid way though
    annealing_cfg.tokens,  # end of annealing (lr = 0)
]

for tokens in annealing_milestones:
    annealing_cfg.tokens_processed = tokens
    lr = annealing_cfg.get_learning_rate(initial_min_lr=initial_cfg.decay_lr_to)
    phase = "Annealing (Linear LR to 0 ↓)"
    if annealing_cfg.tokens_processed == annealing_cfg.tokens:
        phase = "Training complete (LR = 0)"
    print(f"{'Annealing':<15} | Token {annealing_cfg.tokens_processed:<16,d} | {lr:15.8f} | {phase}")


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: Llama 3.1 405B

------ LR Schedule Pre-Training Example For The Llama 3 405B Model ------

Peak LR: 0.00008000 | Min LR: 0.00000080
Initial Stage Steps: 1,200,000
    Warmup ends at step: 8,000
Long-Context Stage Tokens: 800,000,000,000
Annealing Stage Tokens: 40,000,000

------------------------------------------------------------------------------------
Stage           | Metric (Step/Token)    | Learning Rate   | Phase
------------------------------------------------------------------------------------
Initial         | Step 1                 |      0.00000001 | Warmup (Linear LR up ↑)
Initial         | Step 4,000             |      0.00004000 | Warmup (Linear LR up ↑)
Initial         | Step 8,000             |      0.00008000 | Warmup (Linear LR up ↑)
Initial         | Step 600,000           |      0.00004082 | Decay (Cosine LR down ↓)
Initial         | Step 1,199,999         |      0.00000080 | Decay (Cosine LR down ↓)
